In [1]:
import sys
import duckdb
import pandas as pd
from pathlib import Path

print("Python 路径：", sys.executable)
print("DuckDB 版本：", duckdb.__version__)
print("pandas 版本：", pd.__version__)


Python 路径： C:\Users\29443\Documents\实习\ecommerce-conversion\.venv\Scripts\python.exe
DuckDB 版本： 1.5.5
pandas 版本： 3.0.5


In [2]:
data_path = Path("../data/raw/2019-Oct.csv").resolve()

print("数据路径：", data_path)
print("文件存在：", data_path.exists())
print("文件大小：", round(data_path.stat().st_size / 1024**3, 2), "GB")

数据路径： C:\Users\29443\Documents\实习\ecommerce-conversion\data\raw\2019-Oct.csv
文件存在： True
文件大小： 5.28 GB


In [3]:
sql_path = data_path.as_posix()

preview = duckdb.sql(f"""
    SELECT *
    FROM read_csv_auto('{sql_path}')
    LIMIT 5
""").df()

preview

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00,view,44600062,2103807459595387724,NaN,shiseido,35.79,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
1,2019-10-01 00:00:00,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.20,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2,2019-10-01 00:00:01,view,17200506,2053013559792632471,furniture.living_room.sofa,NaN,543.10,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8
3,2019-10-01 00:00:01,view,1307067,2053013558920217191,computers.notebook,lenovo,251.74,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713
4,2019-10-01 00:00:04,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.98,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d


In [4]:
parquet_path = Path("../data/processed/events_2019_10.parquet").resolve()

print("输出位置：", parquet_path)

if parquet_path.exists():
    print("Parquet 文件已经存在，跳过转换。")
else:
    con = duckdb.connect()

    con.execute(f"""
        COPY (
            SELECT *
            FROM read_csv_auto('{data_path.as_posix()}')
        )
        TO '{parquet_path.as_posix()}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
    """)

    con.close()
    print("转换完成。")

print("文件存在：", parquet_path.exists())

if parquet_path.exists():
    print(
        "Parquet 大小：",
        round(parquet_path.stat().st_size / 1024**3, 2),
        "GB"
    )

输出位置： C:\Users\29443\Documents\实习\ecommerce-conversion\data\processed\events_2019_10.parquet
转换完成。
文件存在： True
Parquet 大小： 0.7 GB


## 1. 数据规模与时间范围

In [5]:
parquet_sql_path = parquet_path.as_posix()

overview = duckdb.sql(f"""
    SELECT
        COUNT(*) AS event_count,
        MIN(event_time) AS start_time,
        MAX(event_time) AS end_time,
        COUNT(DISTINCT user_id) AS user_count,
        COUNT(DISTINCT user_session) AS session_count,
        COUNT(DISTINCT product_id) AS product_count,
        COUNT(DISTINCT category_id) AS category_count
    FROM read_parquet('{parquet_sql_path}')
""").df()

overview.T

,0
event_count,42448764
start_time,2019-10-01 00:00:00
end_time,2019-10-31 23:59:59
user_count,3022290
session_count,9244421
product_count,166794
category_count,624


## 2. 用户行为类型分布

In [6]:
event_distribution = duckdb.sql(f"""
    SELECT
        event_type,
        COUNT(*) AS event_count,
        ROUND(
            COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
            2
        ) AS event_share_pct,
        COUNT(DISTINCT user_id) AS user_count,
        COUNT(DISTINCT user_session) AS session_count
    FROM read_parquet('{parquet_sql_path}')
    GROUP BY event_type
    ORDER BY event_count DESC
""").df()

event_distribution

,event_type,event_count,event_share_pct,user_count,session_count
0,view,40779399,96.07,3022130,9242653
1,cart,926516,2.18,337117,573097
2,purchase,742849,1.75,347118,629560


### 初步发现

- 2019 年 10 月共有 4244.88 万条行为记录，其中浏览行为占 96.07%，加购占 2.18%，购买占 1.75%。
- 数据实际只包含 `view`、`cart` 和 `purchase` 三种行为，没有 `remove_from_cart`，后续不进行移除购物车分析。
- 购买用户数和购买会话数均高于加购对应数量，说明购买行为不一定能在当前会话或当前月份中找到对应的加购记录。
- 当前统计仅反映行为分布，不能直接解释为严格的浏览、加购和购买转化率。

## 3. 数据质量检查

In [7]:
missing_summary = duckdb.sql(f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE event_time IS NULL
        ) AS event_time_missing,

        COUNT(*) FILTER (
            WHERE event_type IS NULL
        ) AS event_type_missing,

        COUNT(*) FILTER (
            WHERE product_id IS NULL
        ) AS product_id_missing,

        COUNT(*) FILTER (
            WHERE category_id IS NULL
        ) AS category_id_missing,

        COUNT(*) FILTER (
            WHERE category_code IS NULL
        ) AS category_code_missing,

        COUNT(*) FILTER (
            WHERE brand IS NULL
        ) AS brand_missing,

        COUNT(*) FILTER (
            WHERE price IS NULL
        ) AS price_missing,

        COUNT(*) FILTER (
            WHERE user_id IS NULL
        ) AS user_id_missing,

        COUNT(*) FILTER (
            WHERE user_session IS NULL
        ) AS user_session_missing

    FROM read_parquet('{parquet_sql_path}')
""").df()

missing_summary.T

,0
total_rows,42448764
event_time_missing,0
event_type_missing,0
product_id_missing,0
category_id_missing,0
category_code_missing,13515609
brand_missing,6113008
price_missing,0
user_id_missing,0
user_session_missing,2


### 缺失值检查结论

- 时间、行为类型、商品 ID、品类 ID、价格和用户 ID 等核心字段没有缺失，可以支持后续用户行为分析。
- `category_code` 缺失 13,515,609 行，约占 31.84%，但对应的 `category_id` 完整，因此不删除这些记录；展示时将缺失名称标记为 `unknown`。
- `brand` 缺失 6,113,008 行，约占 14.40%，后续品牌分析将缺失品牌单独归类为 `unknown`。
- `user_session` 仅缺失两行，需要进一步检查；会话分析时可考虑排除，但用户级分析仍可保留。

In [8]:
#检查会话id缺失
missing_session_rows = duckdb.sql(f"""
    SELECT *
    FROM read_parquet('{parquet_sql_path}')
    WHERE user_session IS NULL
""").df()

missing_session_rows

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-06 14:26:10,cart,1801723,2053013554415534427,electronics.video.tv,tcl,135.65,557388939,None
1,2019-10-25 10:36:14,cart,1004767,2053013555631882655,electronics.smartphone,samsung,246.52,549825742,None


### 会话 ID 缺失处理

数据中有两条加购记录缺少 `user_session`，占全部记录约 0.0000047%。
由于无法将其归入具体会话，且删除对整体分析影响可以忽略，因此在清洗后的分析数据中删除这两条记录。原始数据保持不变。

In [10]:
#检查价格异常
price_summary = duckdb.sql(f"""
    SELECT
        MIN(price) AS min_price,
        MAX(price) AS max_price,
        ROUND(AVG(price), 2) AS mean_price,
        ROUND(MEDIAN(price), 2) AS median_price,
        ROUND(QUANTILE_CONT(price, 0.95), 2) AS price_p95,
        ROUND(QUANTILE_CONT(price, 0.99), 2) AS price_p99,

        COUNT(*) FILTER (
            WHERE price < 0
        ) AS negative_price_count,

        COUNT(*) FILTER (
            WHERE price = 0
        ) AS zero_price_count

    FROM read_parquet('{parquet_sql_path}')
""").df()

price_summary.T

,0
min_price,0.00
max_price,2574.07
mean_price,290.32
median_price,162.93
price_p95,1010.25
price_p99,1741.34
negative_price_count,0.00
zero_price_count,68673.00


### 价格分布初步结论

- 数据中没有负价格。
- 商品事件价格中位数为 162.93，平均数为 290.32，说明价格分布明显右偏。
- 最高价格为 2574.07，与第 99 百分位数 1741.34 相比没有达到明显不合理的程度，因此暂不作为异常值删除。
- 共有 68,673 条零价格记录，约占全部记录的 0.16%，需要进一步检查其行为类型和商品分布。

In [11]:
#分析零价格记录
zero_price_by_event = duckdb.sql(f"""
    SELECT
        event_type,
        COUNT(*) AS event_count,
        COUNT(DISTINCT product_id) AS product_count,
        COUNT(DISTINCT user_id) AS user_count,
        COUNT(DISTINCT user_session) AS session_count
    FROM read_parquet('{parquet_sql_path}')
    WHERE price = 0
    GROUP BY event_type
    ORDER BY event_count DESC
""").df()

zero_price_by_event

,event_type,event_count,product_count,user_count,session_count
0,view,68523,11095,33053,38224
1,cart,150,73,91,102


### 零价格处理

零价格记录共 68,673 条，其中 68,523 条为浏览、150 条为加购，没有购买记录。

因此，零价格不会影响实际销售额。为保留有效的用户行为，漏斗和活跃度分析保留这些记录；价格区间分析排除 `price = 0` 的记录。后续建模时将零价格作为价格信息异常标记处理。

In [12]:
#检查完全重复记录
duplicate_summary = duckdb.sql(f"""
    WITH duplicate_groups AS (
        SELECT
            *,
            COUNT(*) AS duplicate_count
        FROM read_parquet('{parquet_sql_path}')
        GROUP BY ALL
        HAVING COUNT(*) > 1
    )

    SELECT
        COUNT(*) AS duplicate_group_count,
        SUM(duplicate_count) AS rows_in_duplicate_groups,
        SUM(duplicate_count - 1) AS possible_extra_rows
    FROM duplicate_groups
""").df()

duplicate_summary.T

,0
duplicate_group_count,18099.0
rows_in_duplicate_groups,48319.0
possible_extra_rows,30220.0


In [13]:
#查看重复行为的行为分步### 重复记录解释

30,220 条可能多余的记录中：

- 加购行为（`cart`）：28,073 条，约占 92.9%；
- 浏览行为（`view`）：2,071 条，约占 6.9%；
- 购买行为（`purchase`）：76 条，约占 0.3%。

加购记录最多完全相同地出现了 41 次，可能的原因包括：

1. 用户连续增加同一商品的购买数量；
2. 前端重复发送加购事件；
3. 数据采集系统重复上报。

由于原始数据中没有订单 ID 和商品数量字段，无法判断完全相同的记录是真实的重复操作，还是日志重复上报。

购买行为中可能重复的记录只有 76 条，且同一记录最多出现两次。即使这些记录全部属于日志错误，对整体结果的影响也很小；但它们也可能代表用户购买了多件同款商品，因此不直接删除。

### 最终处理决定

- 保留全部完全重复记录；
- 用户级漏斗使用 `COUNT(DISTINCT user_id)`，避免重复计算用户；
- 会话级漏斗使用 `COUNT(DISTINCT user_session)`，避免重复计算会话；
- 在项目局限性中说明无法完全区分重复上报和多件购买；
- 当前阶段不再对重复记录进行进一步处理，后续如有需要可进行去重前后的敏感性分析。
duplicate_by_event = duckdb.sql(f"""
    WITH duplicate_groups AS (
        SELECT
            event_type,
            COUNT(*) AS duplicate_count
        FROM read_parquet('{parquet_sql_path}')
        GROUP BY
            event_time,
            event_type,
            product_id,
            category_id,
            category_code,
            brand,
            price,
            user_id,
            user_session
        HAVING COUNT(*) > 1
    )

    SELECT
        event_type,
        COUNT(*) AS duplicate_group_count,
        SUM(duplicate_count) AS rows_in_duplicate_groups,
        SUM(duplicate_count - 1) AS possible_extra_rows,
        MAX(duplicate_count) AS max_repeat_count
    FROM duplicate_groups
    GROUP BY event_type
    ORDER BY possible_extra_rows DESC
""").df()

duplicate_by_event

,event_type,duplicate_group_count,rows_in_duplicate_groups,possible_extra_rows,max_repeat_count
0,cart,15994,44067.0,28073.0,41
1,view,2029,4100.0,2071.0,4
2,purchase,76,152.0,76.0,2


### 重复记录解释

30,220 条可能多余的记录中：

- 加购行为（`cart`）：28,073 条，约占 92.9%；
- 浏览行为（`view`）：2,071 条，约占 6.9%；
- 购买行为（`purchase`）：76 条，约占 0.3%。

加购记录最多完全相同地出现了 41 次，可能的原因包括：

1. 用户连续增加同一商品的购买数量；
2. 前端重复发送加购事件；
3. 数据采集系统重复上报。

由于原始数据中没有订单 ID 和商品数量字段，无法判断完全相同的记录是真实的重复操作，还是日志重复上报。

购买行为中可能重复的记录只有 76 条，且同一记录最多出现两次。即使这些记录全部属于日志错误，对整体结果的影响也很小；但它们也可能代表用户购买了多件同款商品，因此不直接删除。

### 最终处理决定

- 保留全部完全重复记录；
- 用户级漏斗使用 `COUNT(DISTINCT user_id)`，避免重复计算用户；
- 会话级漏斗使用 `COUNT(DISTINCT user_session)`，避免重复计算会话；
- 在项目局限性中说明无法完全区分重复上报和多件购买；
- 当前阶段不再对重复记录进行进一步处理，后续如有需要可进行去重前后的敏感性分析。

In [14]:
#生成清洗后的分析数据
clean_path = Path(
    "../data/processed/events_2019_10_clean.parquet"
).resolve()

if clean_path.exists():
    print("清洗数据已经存在，跳过生成。")
else:
    con = duckdb.connect()

    con.execute(f"""
        COPY (
            SELECT
                event_time,
                event_type,
                product_id,
                category_id,
                COALESCE(category_code, 'unknown') AS category_code,
                COALESCE(brand, 'unknown') AS brand,
                price,
                price = 0 AS is_zero_price,
                user_id,
                user_session
            FROM read_parquet('{parquet_sql_path}')
            WHERE user_session IS NOT NULL
        )
        TO '{clean_path.as_posix()}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
    """)

    con.close()
    print("清洗数据生成完成。")

print("文件存在：", clean_path.exists())

if clean_path.exists():
    print(
        "文件大小：",
        round(clean_path.stat().st_size / 1024**3, 2),
        "GB"
    )

清洗数据生成完成。
文件存在： True
文件大小： 0.71 GB


## 4. 清洗结果验证

In [15]:
clean_sql_path = clean_path.as_posix()

clean_validation = duckdb.sql(f"""
    WITH raw_stats AS (
        SELECT
            COUNT(*) AS raw_rows
        FROM read_parquet('{parquet_sql_path}')
    ),

    clean_stats AS (
        SELECT
            COUNT(*) AS clean_rows,

            COUNT(*) FILTER (
                WHERE user_session IS NULL
            ) AS session_missing,

            COUNT(*) FILTER (
                WHERE category_code IS NULL
            ) AS category_code_missing,

            COUNT(*) FILTER (
                WHERE category_code = 'unknown'
            ) AS category_code_unknown,

            COUNT(*) FILTER (
                WHERE brand IS NULL
            ) AS brand_missing,

            COUNT(*) FILTER (
                WHERE brand = 'unknown'
            ) AS brand_unknown,

            COUNT(*) FILTER (
                WHERE is_zero_price
            ) AS zero_price_count

        FROM read_parquet('{clean_sql_path}')
    )

    SELECT
        raw_rows,
        clean_rows,
        raw_rows - clean_rows AS removed_rows,
        session_missing,
        category_code_missing,
        category_code_unknown,
        brand_missing,
        brand_unknown,
        zero_price_count
    FROM raw_stats
    CROSS JOIN clean_stats
""").df()

clean_validation.T

,0
raw_rows,42448764
clean_rows,42448762
removed_rows,2
session_missing,0
category_code_missing,0
category_code_unknown,13515609
brand_missing,0
brand_unknown,6113008
zero_price_count,68673


### 清洗结果验证结论

清洗后的数据共有 42,448,762 条记录，相比原始数据仅删除了两条缺少会话 ID 的记录。

清洗结果符合预期：

- `user_session` 不再存在缺失值；
- 缺失的 `category_code` 已统一标记为 `unknown`；
- 缺失的 `brand` 已统一标记为 `unknown`；
- 68,673 条零价格记录得到保留，并通过 `is_zero_price` 字段进行标记；
- 完全重复记录保持不变。

清洗后的 Parquet 文件将作为后续业务分析和机器学习的数据源，原始 CSV 和未经清洗的 Parquet 文件保持不变。